<a href="https://colab.research.google.com/github/tanuiivy/bbt4106-202604-D4-classification/blob/feature%2Feda-ivy/bbt4106_202604_D4_classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Predicting Invoice Payment Risk: A Classification Analysis of B2B Client Payment Behavior**

## 1. Setup & dependencies

In [1]:
#For resampling and `shap` for explainability.
!pip install -q imbalanced-learn shap

In [2]:
# Setup & Dependencies
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.model_selection import train_test_split
import urllib.request
import os


# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
sns.set_style('whitegrid')

TARGET = 'payment_risk_category'

## 2. Data loading & exploratory data analysis

In [3]:
# Load the dataset
dataset_path = './data/invoice_payment_risk.csv'
url = 'https://raw.githubusercontent.com/tanuiivy/bbt4106-202604-D4-classification/refs/heads/main/invoice_payment_risk.csv'

if not os.path.exists(dataset_path):
    print("Downloading dataset...")
    if not os.path.exists('./data'):
        os.makedirs('./data')
    urllib.request.urlretrieve(url, dataset_path)
    print("Dataset downloaded✅ ")
else:
    print("Dataset already exists locally✅ ")

invoice_data = pd.read_csv(dataset_path, encoding='utf-8')
print(f"Shape: {invoice_data.shape[0]} rows, {invoice_data.shape[1]} columns")
invoice_data.head()

Dataset downloaded✅ 
Shape: 1500 rows, 16 columns


,invoice_id,client_industry,client_credit_rating,client_company_size,payment_method,has_dispute_history,collateral_or_guarantee,payment_terms_days,invoice_amount_kes,client_relationship_years,prior_late_payments_count,discount_offered_pct,days_outstanding_avg_historical,guarantee_value_kes,economic_sector_growth_rate_pct,payment_risk_category
0,1,Technology,Good,Large,Bank Transfer,Yes,No,45,256337.0,1.9,1,2.73,22.2,NaN,4.87,Late
1,2,Manufacturing,Fair,Small,Bank Transfer,No,No,60,608908.0,6.8,1,2.10,34.0,NaN,2.09,On-Time
2,3,Technology,Good,Medium,Bank Transfer,Yes,No,15,499261.0,5.0,1,2.19,24.6,NaN,4.80,On-Time
3,4,Technology,Good,Medium,Mobile Money,No,No,15,219366.0,8.1,0,1.30,10.3,NaN,4.24,On-Time
4,5,Retail,Excellent,Large,Cheque,Yes,No,45,288282.0,4.5,0,3.26,12.9,NaN,1.42,On-Time


###2.1  Confirm column data types

In [4]:
invoice_data.dtypes

,0
invoice_id,int64
client_industry,object
client_credit_rating,object
client_company_size,object
payment_method,object
has_dispute_history,object
collateral_or_guarantee,object
payment_terms_days,int64
invoice_amount_kes,float64
client_relationship_years,float64


In [5]:
object_cols = invoice_data.select_dtypes(include='object').columns.tolist()
print(f"""
{len(object_cols)} columns are read as object (text) type: {object_cols}.
has_dispute_history and collateral_or_guarantee are Yes/No fields but load as text, not boolean,
because pandas does not auto-detect two-valued strings as booleans. These, along with the other
categorical columns, will need encoding before any model in section 5 can use them.
""")


7 columns are read as object (text) type: ['client_industry', 'client_credit_rating', 'client_company_size', 'payment_method', 'has_dispute_history', 'collateral_or_guarantee', 'payment_risk_category'].
has_dispute_history and collateral_or_guarantee are Yes/No fields but load as text, not boolean,
because pandas does not auto-detect two-valued strings as booleans. These, along with the other
categorical columns, will need encoding before any model in section 5 can use them.



###2.2 Check target class balance

In [6]:
invoice_data['payment_risk_category'].value_counts()

,count
payment_risk_category,
On-Time,975
Late,375
Severely Late,150


In [7]:
invoice_data['payment_risk_category'].value_counts(normalize=True) * 100

,proportion
payment_risk_category,
On-Time,65.0
Late,25.0
Severely Late,10.0


In [8]:
pct = invoice_data['payment_risk_category'].value_counts(normalize=True) * 100
majority_class = pct.idxmax()
minority_class = pct.idxmin()
print(f"""
{majority_class} is the largest class at {pct.max():.1f}% of invoices, while {minority_class}
is the smallest at {pct.min():.1f}%. This gap confirms payment_risk_category is imbalanced,
which is why the train/test split in section 3 needs to be stratified, and why cross-validation
in section 6 needs stratified k-fold rather than plain k-fold.
""")


On-Time is the largest class at 65.0% of invoices, while Severely Late
is the smallest at 10.0%. This gap confirms payment_risk_category is imbalanced,
which is why the train/test split in section 3 needs to be stratified, and why cross-validation
in section 6 needs stratified k-fold rather than plain k-fold.



###2.3 Check categorical frequencies

In [9]:
categorical_cols = ['client_industry', 'client_credit_rating', 'client_company_size',
                     'payment_method', 'has_dispute_history', 'collateral_or_guarantee']

for col in categorical_cols:
    print(f"\n{col}:")
    print(invoice_data[col].value_counts())


client_industry:
client_industry
Manufacturing    394
Retail           334
Technology       286
Construction     255
Healthcare       231
Name: count, dtype: int64

client_credit_rating:
client_credit_rating
Good         537
Fair         453
Excellent    319
Poor         191
Name: count, dtype: int64

client_company_size:
client_company_size
Medium        528
Small         467
Large         358
Enterprise    147
Name: count, dtype: int64

payment_method:
payment_method
Bank Transfer       694
Mobile Money        371
Letter of Credit    229
Cheque              206
Name: count, dtype: int64

has_dispute_history:
has_dispute_history
No     1238
Yes     262
Name: count, dtype: int64

collateral_or_guarantee:
collateral_or_guarantee
No     962
Yes    538
Name: count, dtype: int64


In [10]:
for col in categorical_cols:
    top_category = invoice_data[col].value_counts().idxmax()
    top_pct = invoice_data[col].value_counts(normalize=True).max() * 100
    print(f"{col}: {top_category} is the most common value, at {top_pct:.1f}% of rows.")

client_industry: Manufacturing is the most common value, at 26.3% of rows.
client_credit_rating: Good is the most common value, at 35.8% of rows.
client_company_size: Medium is the most common value, at 35.2% of rows.
payment_method: Bank Transfer is the most common value, at 46.3% of rows.
has_dispute_history: No is the most common value, at 82.5% of rows.
collateral_or_guarantee: No is the most common value, at 64.1% of rows.


In [11]:
numeric_cols = ['payment_terms_days', 'invoice_amount_kes', 'client_relationship_years',
                 'prior_late_payments_count', 'discount_offered_pct',
                 'days_outstanding_avg_historical', 'guarantee_value_kes',
                 'economic_sector_growth_rate_pct']

invoice_data[numeric_cols].describe()

,payment_terms_days,invoice_amount_kes,client_relationship_years,prior_late_payments_count,discount_offered_pct,days_outstanding_avg_historical,guarantee_value_kes,economic_sector_growth_rate_pct
count,1500.000000,1.500000e+03,1500.000000,1500.000000,1500.000000,1500.000000,5.380000e+02,1500.000000
mean,41.860000,3.627590e+05,5.833267,1.090000,2.030087,24.903600,2.898271e+05,2.326747
std,21.249504,2.534686e+05,4.062270,1.033414,1.145037,11.880925,2.245551e+05,1.316062
min,15.000000,1.000000e+04,0.200000,0.000000,0.000000,0.000000,1.178712e+04,-0.120000
25%,30.000000,1.841765e+05,2.900000,0.000000,1.170000,16.300000,1.405622e+05,1.310000
50%,45.000000,3.044655e+05,4.800000,1.000000,2.010000,23.550000,2.315444e+05,2.040000
75%,45.000000,4.768210e+05,7.800000,2.000000,2.782500,31.200000,3.870093e+05,3.150000
max,90.000000,1.991781e+06,25.000000,6.000000,5.550000,78.000000,1.487066e+06,5.460000


In [12]:
print(f"""
invoice_amount_kes ranges from KES {invoice_data['invoice_amount_kes'].min():,.0f}
to KES {invoice_data['invoice_amount_kes'].max():,.0f}, with a mean of
KES {invoice_data['invoice_amount_kes'].mean():,.0f} against a median of
KES {invoice_data['invoice_amount_kes'].median():,.0f}. The mean sitting above the median
is an early signal of right skew, checked formally in 2.4.
""")


invoice_amount_kes ranges from KES 10,000
to KES 1,991,781, with a mean of
KES 362,759 against a median of
KES 304,466. The mean sitting above the median
is an early signal of right skew, checked formally in 2.4.



###2.4 Compute numeric distribution measures

In [13]:
q1 = invoice_data[numeric_cols].quantile(0.25)
q2 = invoice_data[numeric_cols].quantile(0.50)
q3 = invoice_data[numeric_cols].quantile(0.75)
iqr = q3 - q1

distribution_summary = pd.DataFrame({
    'min': invoice_data[numeric_cols].min(),
    'q1_25%': q1,
    'q2_50%_median': q2,
    'q3_75%': q3,
    'max': invoice_data[numeric_cols].max(),
    'mean': invoice_data[numeric_cols].mean(),
    'mode': invoice_data[numeric_cols].mode().iloc[0],
    'variance': invoice_data[numeric_cols].var(),
    'std_dev': invoice_data[numeric_cols].std(),
    'skewness': invoice_data[numeric_cols].skew(),
    'kurtosis': invoice_data[numeric_cols].kurtosis(),
    'IQR': iqr,
    'lower_fence': q1 - 1.5 * iqr,
    'upper_fence': q3 + 1.5 * iqr,
})

distribution_summary.round(2)

,min,q1_25%,q2_50%_median,q3_75%,max,mean,mode,variance,std_dev,skewness,kurtosis,IQR,lower_fence,upper_fence
payment_terms_days,15.00,30.00,45.00,45.00,90.00,41.86,30.00,4.515400e+02,21.25,0.87,0.20,15.00,7.50,67.50
invoice_amount_kes,10000.00,184176.50,304465.50,476821.00,1991781.00,362759.00,10000.00,6.424634e+10,253468.62,1.49,3.29,292644.50,-254790.25,915787.75
client_relationship_years,0.20,2.90,4.80,7.80,25.00,5.83,3.90,1.650000e+01,4.06,1.41,2.71,4.90,-4.45,15.15
prior_late_payments_count,0.00,0.00,1.00,2.00,6.00,1.09,1.00,1.070000e+00,1.03,1.01,1.07,2.00,-3.00,5.00
discount_offered_pct,0.00,1.17,2.01,2.78,5.55,2.03,0.00,1.310000e+00,1.15,0.22,-0.42,1.61,-1.25,5.20
days_outstanding_avg_historical,0.00,16.30,23.55,31.20,78.00,24.90,21.10,1.411600e+02,11.88,0.85,1.09,14.90,-6.05,53.55
guarantee_value_kes,11787.12,140562.20,231544.36,387009.33,1487066.36,289827.08,11787.12,5.042498e+10,224555.08,1.70,3.81,246447.12,-229108.48,756680.01
economic_sector_growth_rate_pct,-0.12,1.31,2.04,3.15,5.46,2.33,1.53,1.730000e+00,1.32,0.57,-0.69,1.84,-1.45,5.91


In [14]:
outlier_mask = (
    (invoice_data[numeric_cols] < (q1 - 1.5 * iqr)) |
    (invoice_data[numeric_cols] > (q3 + 1.5 * iqr))
)

outlier_counts = outlier_mask.sum().rename('outlier_count').to_frame()
outlier_counts

,outlier_count
payment_terms_days,151
invoice_amount_kes,58
client_relationship_years,47
prior_late_payments_count,1
discount_offered_pct,5
days_outstanding_avg_historical,35
guarantee_value_kes,26
economic_sector_growth_rate_pct,0


In [15]:
most_skewed = distribution_summary['skewness'].abs().idxmax()
most_outliers = outlier_counts['outlier_count'].idxmax()

print(f"""
{most_skewed} is the most skewed numeric column (skewness = {distribution_summary.loc[most_skewed, 'skewness']:.2f}),
consistent with its mean (KES {distribution_summary.loc[most_skewed, 'mean']:,.0f}) sitting above
its median (KES {distribution_summary.loc[most_skewed, 'q2_50%_median']:,.0f}).

{most_outliers} has the most flagged outliers under the 1.5x IQR rule, at
{outlier_counts.loc[most_outliers, 'outlier_count']} rows. This is misleading rather than a data
quality concern: payment_terms_days only takes a small set of fixed values (15, 30, 45, 60, 90),
so the IQR fence is reacting to a common discrete tier, not a genuine anomaly. The remaining
flagged columns (invoice_amount_kes, client_relationship_years, days_outstanding_avg_historical,
guarantee_value_kes) are more meaningful outlier candidates, since they're genuinely continuous,
and warrant inspection rather than automatic removal in section 5.
""")


guarantee_value_kes is the most skewed numeric column (skewness = 1.70),
consistent with its mean (KES 289,827) sitting above
its median (KES 231,544).

payment_terms_days has the most flagged outliers under the 1.5x IQR rule, at
151 rows. This is misleading rather than a data
quality concern: payment_terms_days only takes a small set of fixed values (15, 30, 45, 60, 90),
so the IQR fence is reacting to a common discrete tier, not a genuine anomaly. The remaining
flagged columns (invoice_amount_kes, client_relationship_years, days_outstanding_avg_historical,
guarantee_value_kes) are more meaningful outlier candidates, since they're genuinely continuous,
and warrant inspection rather than automatic removal in section 5.



###2.5 Check missingness

In [16]:
invoice_data.isnull().sum()

,0
invoice_id,0
client_industry,0
client_credit_rating,0
client_company_size,0
payment_method,0
has_dispute_history,0
collateral_or_guarantee,0
payment_terms_days,0
invoice_amount_kes,0
client_relationship_years,0


In [17]:
missing_guarantee = invoice_data['guarantee_value_kes'].isnull().sum()
no_collateral = (invoice_data['collateral_or_guarantee'] == 'No').sum()
print(f"""
guarantee_value_kes is missing in {missing_guarantee} of {len(invoice_data)} rows
({missing_guarantee / len(invoice_data) * 100:.1f}%). This matches the {no_collateral} rows
where collateral_or_guarantee is No — the value is absent because there is no guarantee to
report, not because of a data collection failure. This is structural missingness, not random
missingness, and needs to be imputed differently from any genuinely random gaps in section 5.
""")


guarantee_value_kes is missing in 962 of 1500 rows
(64.1%). This matches the 962 rows
where collateral_or_guarantee is No — the value is absent because there is no guarantee to
report, not because of a data collection failure. This is structural missingness, not random
missingness, and needs to be imputed differently from any genuinely random gaps in section 5.



###2.6 Measures of relationship

In [18]:
from sklearn.feature_selection import f_classif

numeric_predictors_filled = invoice_data[numeric_cols].fillna(invoice_data[numeric_cols].median())

f_stats, p_values = f_classif(numeric_predictors_filled, invoice_data[TARGET])
anova_summary = pd.DataFrame({
    'feature': numeric_cols,
    'F_statistic': f_stats,
    'p_value': p_values
}).sort_values('F_statistic', ascending=False)

anova_summary.round(4)

,feature,F_statistic,p_value
0,payment_terms_days,148.4225,0.0000
3,prior_late_payments_count,70.1130,0.0000
5,days_outstanding_avg_historical,60.3109,0.0000
7,economic_sector_growth_rate_pct,15.8523,0.0000
4,discount_offered_pct,1.2006,0.3013
6,guarantee_value_kes,1.0433,0.3525
1,invoice_amount_kes,0.1485,0.8620
2,client_relationship_years,0.0992,0.9055


In [19]:
def cramers_v(confusion_matrix):
    chi2 = stats.chi2_contingency(confusion_matrix)[0]
    n = confusion_matrix.sum().sum()
    phi2 = chi2 / n
    r, k = confusion_matrix.shape
    phi2_corrected = max(0, phi2 - ((k - 1) * (r - 1)) / (n - 1))
    r_corrected = r - ((r - 1) ** 2) / (n - 1)
    k_corrected = k - ((k - 1) ** 2) / (n - 1)
    return np.sqrt(phi2_corrected / min(k_corrected - 1, r_corrected - 1))

cramers_v_results = {}
for col in categorical_cols:
    contingency = pd.crosstab(invoice_data[col], invoice_data[TARGET])
    cramers_v_results[col] = cramers_v(contingency.values)

cramers_v_summary = pd.DataFrame({
    'feature': list(cramers_v_results.keys()),
    'cramers_v': list(cramers_v_results.values())
}).sort_values('cramers_v', ascending=False)

cramers_v_summary.round(4)

,feature,cramers_v
1,client_credit_rating,0.2873
4,has_dispute_history,0.1921
0,client_industry,0.1171
3,payment_method,0.0469
2,client_company_size,0.0185
5,collateral_or_guarantee,0.0166


`client_credit_rating` shows the strongest categorical association with `payment_risk_category`
(Cramer's V = 0.287), which falls in the moderate range (0.20-0.40), meaning a client's credit
rating carries real information about their payment risk. `has_dispute_history` follows at 0.192
(weak association), and `client_industry` at 0.117. `payment_method`, `client_company_size`, and
`collateral_or_guarantee` all fall below 0.05, showing negligible association with risk category
on their own.

Among numeric predictors, `payment_terms_days` shows by far the strongest relationship with the
target (F-statistic = 148.42, p < 0.001), followed by `prior_late_payments_count` (F = 70.11)
and `days_outstanding_avg_historical` (F = 60.31), both significant at p < 0.001. This is
consistent with the hypothesis in 2.7 that payment history drives risk more than invoice-specific
details. `economic_sector_growth_rate_pct` is also significant (F = 15.85), though weaker. Four
features, `discount_offered_pct`, `guarantee_value_kes`, `invoice_amount_kes`, and
`client_relationship_years`, show p-values well above 0.05, meaning no evidence of a linear,
mean-based relationship with risk category in isolation. This does not rule them out entirely:
ANOVA only detects linear separation, so section 4's mutual information check may still surface
non-linear relationships these results miss.


### 2.7 Hypothesis

Based on the measures of relationship in 2.6, `payment_terms_days`, `prior_late_payments_count`,
and `days_outstanding_avg_historical` are expected to be the strongest predictors of
`payment_risk_category`, given their high ANOVA F-statistics (148.42, 70.11, and 60.31
respectively, all p < 0.001). Among categorical features, `client_credit_rating` (Cramer's V =
0.287) is expected to contribute the most, with `has_dispute_history` (0.192) as a secondary
signal.

This revises an earlier assumption that payment history alone would dominate: `payment_terms_days`
— an invoice-specific term set at issuance, not part of a client's payment track record — shows
the strongest relationship of any single feature. `client_relationship_years`, despite intuitively
seeming like a track-record signal, shows no meaningful relationship with risk category
(F = 0.10, p = 0.91) and is not expected to contribute much on its own. This is tested formally
in section 6.

## 3. Train/test split

#### 3.1 The split

In [20]:
# payment_risk_category is imbalanced (Severely Late is the minority class, confirmed in 2.2).
# A plain random split risks leaving too few Severely Late invoices in the test set to evaluate
# properly. Stratifying directly on the target ensures both sets keep the same class proportions.

X = invoice_data.drop(columns=['invoice_id', TARGET])
y = invoice_data[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set: {X_train.shape[0]} rows")
print(f"Test set: {X_test.shape[0]} rows")

Training set: 1200 rows
Test set: 300 rows


In [21]:
train_pct = y_train.value_counts(normalize=True) * 100
test_pct = y_test.value_counts(normalize=True) * 100

split_check = pd.DataFrame({'train_%': train_pct, 'test_%': test_pct}).round(1)
print(split_check)


                       train_%  test_%
payment_risk_category                 
On-Time                   65.0    65.0
Late                      25.0    25.0
Severely Late             10.0    10.0


In [22]:
print(f"""
An 80/20 split was used given the dataset's size (1,500 rows) — large enough to hold out 20%
({X_test.shape[0]} rows) while still leaving {X_train.shape[0]} rows to train on. The table above
confirms stratification worked as intended: train and test class percentages are close to identical
for On-Time, Late, and Severely Late, so the minority class is represented in both sets rather than
concentrated in one.
""")


An 80/20 split was used given the dataset's size (1,500 rows) — large enough to hold out 20%
(300 rows) while still leaving 1200 rows to train on. The table above
confirms stratification worked as intended: train and test class percentages are close to identical
for On-Time, Late, and Severely Late, so the minority class is represented in both sets rather than
concentrated in one.



## 4. Feature selection

## 5. Preprocessing pipeline

## 6. Modeling & cross-validation

## 7. Diagnostics

## 8. Evaluation of candidate models

## 9. Hyperparameter tuning

## 10. Explainability

## 11. Model persistence

## 12. Conclusion & limitations